# Speculative sampling from scratch
**LLMs From Basics · M5.5 hands-on** &nbsp; [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhishekaryan23/LLM-Learner/blob/main/course/notebooks/m5-5-speculative-decoding.ipynb)

A draft model $q$ proposes $k$ tokens; the target $p$ checks them. Each draft token $x$ is kept with probability $\min(1, p(x)/q(x))$;
the first rejected one is replaced by a sample from $\max(0, p - q)$, renormalised. The lesson proved this gives **exactly** the target's distribution.
Here you'll check that claim with a simulation instead of algebra:

1. implement the acceptance probability and the residual distribution
2. implement one full round of speculative sampling
3. verify **empirically** that the outputs follow the target's distribution, and that a "naive" shortcut does not
4. measure the acceptance rate and tokens per step, and compare with $(1-\alpha^{k+1})/(1-\alpha)$
5. experiment with draft quality and draft length

Our "models" are tiny **bigram** tables over an 8-token vocabulary: the next-token distribution depends only on the previous token.
That keeps everything fast and exactly computable. About 25 minutes. Solutions are at the end.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)

V = 8                                                      # vocabulary size
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True); e = np.exp(z); return e / e.sum(axis=axis, keepdims=True)

target_logits = rng.normal(size=(V, V)) * 2.0
P = softmax(target_logits)                                 # P[prev] = target's next-token distribution
def make_draft(blur, noise, seed=1):
    """A cheaper, blurrier imitation of the target: shrink the logits and add noise."""
    r = np.random.default_rng(seed)
    return softmax(target_logits * blur + r.normal(size=(V, V)) * noise)
Q = make_draft(blur=0.6, noise=0.6)                        # Q[prev] = draft's next-token distribution

def sample(dist, rng):
    """Draw one token index from a probability vector (faster than rng.choice for tiny vectors)."""
    return min(int(np.searchsorted(np.cumsum(dist), rng.random() * dist.sum())), len(dist) - 1)

def tv(a, b):
    """Total-variation distance between two distributions: 0 = identical, 1 = disjoint."""
    return 0.5 * np.abs(np.asarray(a) - np.asarray(b)).sum()

print("target P[0]:", P[0]); print("draft  Q[0]:", Q[0])
print("acceptance chance from context 0, sum(min(p, q)) =", np.minimum(P[0], Q[0]).sum().round(3))

## 1. The two rules
`accept_prob(p, q, x)` returns $\min(1, p_x/q_x)$. `residual(p, q)` returns $\max(0, p-q)$ divided by its sum.

In [ ]:
def accept_prob(p, q, x):
    """Probability of keeping draft token x: min(1, p[x] / q[x])."""
    # TODO
    raise NotImplementedError("Fill in accept_prob")

def residual(p, q):
    """The distribution to resample from after a rejection: max(0, p - q), normalised."""
    # TODO: clip p - q at 0 (np.maximum), then divide by the sum
    raise NotImplementedError("Fill in residual")

In [ ]:
# The lesson's worked example: p = [0.5, 0.3, 0.2], q = [0.2, 0.6, 0.2]
p_ex, q_ex = np.array([.5, .3, .2]), np.array([.2, .6, .2])
assert np.isclose(accept_prob(p_ex, q_ex, 1), 0.5) and accept_prob(p_ex, q_ex, 0) == 1.0
assert np.allclose(residual(p_ex, q_ex), [1, 0, 0])
# The faded example: p = [0.5, 0.1, 0.4]
p_f = np.array([.5, .1, .4])
assert np.isclose(accept_prob(p_f, q_ex, 1), 1 / 6) and np.allclose(residual(p_f, q_ex), [0.6, 0, 0.4])

# Exactness, computed analytically: P(out = x) = q(x)·min(1, p/q) + P(reject)·residual(x)
def output_dist(p, q):
    acc = np.array([q[x] * accept_prob(p, q, x) for x in range(len(p))])
    return acc + (1 - acc.sum()) * residual(p, q)
assert np.allclose(output_dist(p_ex, q_ex), p_ex) and np.allclose(output_dist(P[3], Q[3]), P[3])
print("✅ rules work, and one position's output distribution equals p exactly:", output_dist(p_f, q_ex))

## 2. One round of speculative sampling
Implement `speculative_step(last, k, rng)` using the global tables `P` (target) and `Q` (draft). On a GPU, step 2 is **one** batched forward pass of the target;
here it's just a few table lookups.

In [ ]:
def speculative_step(last, k, rng):
    """One round of speculative sampling from context token `last`.
    Returns (new_tokens, n_accepted): between 1 and k+1 new tokens, and how many drafts were kept."""
    # TODO 1: the draft proposes k tokens autoregressively: x = sample(Q[ctx], rng), then ctx = x
    # TODO 2: the target's distribution at each of the k+1 positions: P[last], P[x1], ..., P[xk]
    # TODO 3: for each draft x_i (left to right): keep it if rng.random() < accept_prob(p_i, q_i, x_i);
    #         otherwise append sample(residual(p_i, q_i), rng) and return (out, i) immediately
    # TODO 4: if all k were kept, append a bonus token sampled from P[xk] and return (out, k)
    raise NotImplementedError("Fill in speculative_step")

In [ ]:
r = np.random.default_rng(42)
for _ in range(500):
    toks, n_acc = speculative_step(last=2, k=4, rng=r)
    assert 1 <= len(toks) <= 5 and len(toks) == n_acc + 1, (toks, n_acc)
    assert all(0 <= t < V for t in toks)
print("✅ each round returns between 1 and k+1 tokens, e.g.", speculative_step(2, 4, r))

## 3. Is the output really the target's distribution?
Start from context token 0 many times and look at the **first two** generated tokens. Under the target alone, the pair $(a, b)$ has probability
$P[0, a] \cdot P[a, b]$. Compare that with the frequencies speculative sampling produces.
Then try a tempting shortcut, **resampling from $p$ instead of the residual**, and see it fail.

In [ ]:
def first_two(step_fn, n, k=4, seed=7):
    """Empirical distribution of the first two generated tokens (as a V×V table), starting from token 0."""
    r = np.random.default_rng(seed); counts = np.zeros((V, V))
    for _ in range(n):
        toks, _ = step_fn(0, k, r)
        while len(toks) < 2:                      # the round produced only 1 token: run another round
            more, _ = step_fn(toks[-1], k, r); toks = toks + more
        counts[toks[0], toks[1]] += 1
    return counts / n

exact = P[0][:, None] * P                         # target-only joint distribution of the first two tokens
N = 40_000
emp = first_two(speculative_step, N)
print(f"TV(speculative, target) over first-token pairs = {tv(emp, exact):.4f}   (sampling noise alone is about 0.01-0.02)")
assert tv(emp.sum(1), P[0]) < 0.015, "The first token's distribution is off."
assert tv(emp, exact) < 0.03, "The pair distribution is off."

def naive_step(last, k, rng):
    """WRONG on purpose: like speculative_step, but resamples from p instead of the residual."""
    drafts, ctx = [], last
    for _ in range(k):
        x = sample(Q[ctx], rng); drafts.append(x); ctx = x
    contexts = [last] + drafts; out = []
    for i, x in enumerate(drafts):
        if rng.random() < accept_prob(P[contexts[i]], Q[contexts[i]], x):
            out.append(x)
        else:
            out.append(sample(P[contexts[i]], rng)); return out, i
    out.append(sample(P[drafts[-1]], rng)); return out, k

emp_naive = first_two(naive_step, N)
print(f"TV(naive shortcut, target) = {tv(emp_naive, exact):.4f}")
assert tv(emp_naive, exact) > tv(emp, exact) + 0.02
print("✅ speculative sampling matches the target; the shortcut is measurably biased")

## 4. Acceptance rate and tokens per step
Write `expected_tokens(alpha, k)`, then measure the real thing: run many rounds, record how many tokens each produced, and compare.

In [ ]:
def expected_tokens(alpha, k):
    """Expected tokens per target pass: 1 + alpha + ... + alpha^k."""
    # TODO: use the closed form (1 - alpha^(k+1)) / (1 - alpha)
    raise NotImplementedError("Fill in expected_tokens")

In [ ]:
assert np.isclose(expected_tokens(0.8, 4), 3.3616) and np.isclose(expected_tokens(0.6, 4), 2.3056)
assert np.isclose(expected_tokens(0.5, 3), 1.875)

def measure(k, rounds=20_000, seed=3):
    r = np.random.default_rng(seed); last = 0; produced = drafted_kept = 0
    for _ in range(rounds):
        toks, n_acc = speculative_step(last, k, r)
        produced += len(toks); drafted_kept += n_acc; last = toks[-1]
    return produced / rounds, drafted_kept / (rounds * k)

k = 4
tok_per_step, frac_kept = measure(k)
# alpha as defined in the lesson: the average of beta = sum(min(p, q)) over the contexts we visit
stationary = np.linalg.matrix_power(P, 200)[0]          # long-run frequency of each context token
alpha = float(stationary @ np.minimum(P, Q).sum(1))
print(f"alpha ≈ {alpha:.3f}; measured tokens per target pass = {tok_per_step:.2f}; formula = {expected_tokens(alpha, k):.2f}")
print(f"fraction of drafted tokens kept = {frac_kept:.3f}")
assert abs(tok_per_step - expected_tokens(alpha, k)) < 0.35
print("✅ the formula is a good approximation (it assumes every position has the same acceptance chance)")

## 5. Experiment: draft quality and draft length
A sharper draft (closer to the target) raises $\alpha$. Longer drafts give more tokens per pass but cost more draft passes.
Run the cell; then change `c` (the draft's cost relative to the target) and the `blur`/`noise` settings.

In [ ]:
c = 0.05
drafts = {"good draft": dict(blur=0.9, noise=0.2), "ok draft": dict(blur=0.6, noise=0.6), "poor draft": dict(blur=0.2, noise=1.2)}
ks = list(range(1, 9))
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
Q_saved = Q
for name, cfg in drafts.items():
    Q = make_draft(**cfg)                                  # speculative_step reads the global Q
    a = float(stationary @ np.minimum(P, Q).sum(1))
    tps = [measure(kk, rounds=4000)[0] for kk in ks]
    ax[0].plot(ks, tps, "o-", label=f"{name} (α≈{a:.2f})")
    ax[1].plot(ks, [t / (1 + c * kk) for t, kk in zip(tps, ks)], "o-", label=name)
Q = Q_saved
ax[0].set(xlabel="draft length k", ylabel="tokens per target pass", title="measured tokens per pass")
ax[1].set(xlabel="draft length k", ylabel="speedup (×)", title=f"speedup with draft cost c = {c}")
for a_ in ax: a_.grid(alpha=.3); a_.legend()
plt.tight_layout(); plt.show()

**Think about it** (answers below the solutions):
1. Why does the "naive" shortcut over-produce some tokens? Which ones?
2. The draft model must use the same tokenizer as the target. What would break if it didn't?
3. At batch size 64 the target's forward pass is no longer "free" for extra tokens. How would you change the speedup formula?

---
## Solutions
Only run these if you're stuck. They redefine the functions; re-run the check cells afterwards.

In [ ]:
def accept_prob(p, q, x):
    """Probability of keeping draft token x: min(1, p[x] / q[x])."""
    return min(1.0, p[x] / q[x])

def residual(p, q):
    """The distribution to resample from after a rejection: max(0, p - q), normalised."""
    r = np.maximum(p - q, 0.0)
    return r / r.sum()

In [ ]:
def speculative_step(last, k, rng):
    """One round of speculative sampling from context token `last`.
    Returns (new_tokens, n_accepted): between 1 and k+1 new tokens, and how many drafts were kept."""
    # 1. the draft proposes k tokens autoregressively
    drafts, ctx = [], last
    for _ in range(k):
        x = sample(Q[ctx], rng)
        drafts.append(x); ctx = x
    # 2. the target's distributions at all k+1 positions (on a GPU: ONE forward pass)
    contexts = [last] + drafts
    p_dists = [P[c] for c in contexts]
    # 3. accept / reject left to right
    out = []
    for i, x in enumerate(drafts):
        q_i = Q[contexts[i]]
        if rng.random() < accept_prob(p_dists[i], q_i, x):
            out.append(x)
        else:
            out.append(sample(residual(p_dists[i], q_i), rng))
            return out, i
    # 4. all accepted: a bonus token from the target's last distribution
    out.append(sample(p_dists[k], rng))
    return out, k

In [ ]:
def expected_tokens(alpha, k):
    """Expected tokens per target pass: 1 + alpha + ... + alpha^k."""
    return (1 - alpha ** (k + 1)) / (1 - alpha)

**Think-about-it answers**
1. Accepting already produces $\min(p, q)$ of each token's probability. Resampling from $p$ then adds $P(\text{reject})\,p(x)$, so tokens where the draft is already generous ($q \ge p$) get extra probability on top of their full share, and are over-produced. The residual $\max(0, p - q)$ puts the rejected mass only on tokens the draft under-proposed.
2. $p$ and $q$ must be distributions over the **same** set of tokens, and the target must be able to score the draft's tokens directly. With different tokenizers the ratio $p(x)/q(x)$ isn't defined. (Research methods exist for cross-tokenizer drafting, but plain speculative sampling needs a shared vocabulary.)
3. Make the verification cost depend on $k$: if a target pass over $k+1$ tokens costs $T(k)$ relative to one ordinary step, the speedup is $\mathbb{E}[\text{tokens}] / (T(k) + ck)$. At small batch $T(k) \approx 1$; on a busy, compute-bound server $T(k)$ grows with $k$, and the speedup can drop below 1.

**Next:** M5.6, batching and serving metrics: what happens when many users share one GPU.